# Feladat: Adattudománnyal kapcsolatos szöveg elemzése

Ebben a példában végezzünk egy egyszerű gyakorlatot, amely lefedi egy hagyományos adattudományi folyamat összes lépését. Nem kell kódot írnod, egyszerűen rákattinthatsz az alábbi cellákra a végrehajtáshoz és az eredmény megfigyeléséhez. Feladatként ösztönzünk, hogy próbáld ki ezt a kódot különböző adatokkal is.

## Cél

Ebben a leckében az adattudománnyal kapcsolatos különböző fogalmakat tárgyaltuk. Próbáljunk meg több kapcsolódó fogalmat felfedezni **szövegbányászat** segítségével. Egy adattudományról szóló szöveggel kezdünk, kivonjuk belőle a kulcsszavakat, majd megpróbáljuk vizualizálni az eredményt.

Szövegként a Wikipédián található Adattudomány oldalt fogom használni:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## 1. lépés: Az adatok beszerzése

Minden adat tudományi folyamat első lépése az adatok beszerzése. Ehhez a `requests` könyvtárat fogjuk használni:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## 2. lépés: Az adatok átalakítása

A következő lépés az adatok olyan formára való átalakítása, amely feldolgozható. Esetünkben letöltöttük a weboldal HTML forráskódját, és azt sima szöveggé kell alakítanunk.

Erre sokféle módszer létezik. Mi a [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/) nevű, népszerű Python könyvtárat fogjuk használni a HTML elemzésére. A BeautifulSoup lehetővé teszi, hogy célzottan megcélozzuk a HTML elemeket, így a Wikipédia fő cikkének tartalmára koncentrálhatunk, miközben csökkenthetjük a navigációs menük, oldalsávok, láblécek és egyéb lényegtelen tartalmak arányát (bár némi sablonszöveg még maradhat).


Először telepítenünk kell a BeautifulSoup könyvtárat HTML elemzéshez:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## 3. lépés: Elemzések készítése

A legfontosabb lépés, hogy adatainkat olyan formába alakítsuk át, amelyből következtetéseket vonhatunk le. Esetünkben kulcsszavakat szeretnénk kinyerni a szövegből, és megvizsgálni, mely kulcsszavak a legjelentősebbek.

A kulcsszókinyeréshez egy Python könyvtárat, a [RAKE](https://github.com/aneesha/RAKE) használjuk. Először is telepítsük ezt a könyvtárat, ha még nincs telepítve: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

A fő funkcionalitás az `Rake` objektumból érhető el, amelyet bizonyos paraméterek segítségével testre szabhatunk. Esetünkben a kulcsszó minimális hosszát 5 karakterre, a kulcsszó dokumentumban való minimális előfordulását 3-ra, és a kulcsszóban szereplő szavak maximális számát 2-re állítjuk be. Bátran kísérletezzen más értékekkel és figyelje meg az eredményt.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Egy listát kaptunk a kifejezésekről és a hozzájuk tartozó fontossági fokokról. Ahogy látható, a legrelevánsabb tudományágak, mint a gépi tanulás és a big data, a lista élére kerültek.

## 4. lépés: Az eredmény vizualizálása

Az adatok legjobban vizuális formában értelmezhetőek. Ezért gyakran érdemes az adatokat megjeleníteni, hogy betekintéseket nyerjünk. A `matplotlib` Python könyvtárat használhatjuk arra, hogy egyszerűen ábrázoljuk a kulcsszavak eloszlását a relevanciájuk szerint:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Van azonban egy még jobb módja a szósűrűségek megjelenítésének - **szófelhő** használatával. Szükségünk lesz egy másik könyvtár telepítésére, hogy megrajzolhassuk a szófelhőt a kulcsszavainkból.


In [ ]:
!{sys.executable} -m pip install wordcloud

A `WordCloud` objektum feladata, hogy vagy az eredeti szöveget, vagy előre kiszámított szósűrűségi listát fogadjon, és egy képet adjon vissza, amely aztán megjeleníthető a `matplotlib` segítségével:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Eredeti szöveget is átadhatunk a `WordCloud`-nak – nézzük meg, hogy képesek vagyunk-e hasonló eredményt elérni:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Látható, hogy a szófelhő most már lenyűgözőbbnek tűnik, de sok zajt is tartalmaz (például nem kapcsolódó szavakat, mint a `Retrieved on`). Emellett kevesebb kétszavas kulcsszót kapunk, mint például *adat tudós*, vagy *számítástechnika*. Ennek oka, hogy a RAKE algoritmus sokkal jobb munkát végez a jó kulcsszavak kiválasztásában a szövegből. Ez a példa jól illusztrálja az adatelőkészítés és tisztítás fontosságát, mert egy tiszta kép a végén lehetővé teszi számunkra, hogy jobb döntéseket hozzunk.

Ebben a gyakorlatban egy egyszerű folyamaton mentünk keresztül, amely során értelmet nyertünk egy Wikipedia szövegből kulcsszavak és szófelhő formájában. Ez a példa meglehetősen egyszerű, de jól bemutatja az összes tipikus lépést, amelyet egy adatszakértő tesz az adatfeldolgozás során, az adatok beszerzésétől egészen a vizualizációig.

A tanfolyamunk során részletesen meg fogjuk vitatni ezeket a lépéseket. 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
